In [ ]:
# ============================================================
# Single Tail Propagation Story
# ============================================================

import polars as pl
import pandas as pd
import plotly.graph_objects as go

DATA_PATH = "shared-notebooks/data/flights_canonical_2019.parquet"
TAIL_TO_VIEW = None  # set manually like "N12345" or leave None for automatic choice

lf = pl.scan_parquet(DATA_PATH)
schema = lf.collect_schema()

candidate_cols = [
    "Tail_Number",
    "TailNum",
    "Origin",
    "Dest",
    "DepDelay",
    "ArrDelay",
    "dep_ts_actual",
    "arr_ts_actual",
    "prev_arr_delay",
    "turnaround_minutes",
    "rotation_continuity_flag",
    "flight_id",
]
existing_cols = [c for c in candidate_cols if c in schema.names()]
df = lf.select(existing_cols).collect()

if "Tail_Number" in df.columns and "TailNum" not in df.columns:
    df = df.rename({"Tail_Number": "TailNum"})

required = ["TailNum", "Origin", "Dest", "dep_ts_actual"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.with_columns(
    pl.col("dep_ts_actual").cast(pl.Datetime, strict=False),
    pl.col("arr_ts_actual").cast(pl.Datetime, strict=False) if "arr_ts_actual" in df.columns else pl.lit(None),
)

df = df.filter(
    pl.col("TailNum").is_not_null() &
    pl.col("Origin").is_in(airport_coords.keys()) &
    pl.col("Dest").is_in(airport_coords.keys())
)

# choose a tail automatically if not provided
if TAIL_TO_VIEW is None:
    tail_counts = (
        df.group_by("TailNum")
          .agg([
              pl.len().alias("n_flights"),
              pl.mean("DepDelay").alias("mean_dep_delay") if "DepDelay" in df.columns else pl.lit(None).alias("mean_dep_delay")
          ])
          .sort(["n_flights", "mean_dep_delay"], descending=[True, True])
    )
    TAIL_TO_VIEW = tail_counts.row(0)[0]

print("Using tail:", TAIL_TO_VIEW)

story = (
    df.filter(pl.col("TailNum") == TAIL_TO_VIEW)
      .sort("dep_ts_actual")
      .with_row_index("leg_num")
)

pdf = story.to_pandas()

fig = go.Figure()

for i, row in pdf.iterrows():
    lat0, lon0 = airport_coords[row["Origin"]]
    lat1, lon1 = airport_coords[row["Dest"]]

    dep_delay = row["DepDelay"] if "DepDelay" in pdf.columns and pd.notnull(row["DepDelay"]) else None
    prev_arr_delay = row["prev_arr_delay"] if "prev_arr_delay" in pdf.columns and pd.notnull(row["prev_arr_delay"]) else None
    turnaround = row["turnaround_minutes"] if "turnaround_minutes" in pdf.columns and pd.notnull(row["turnaround_minutes"]) else None

    hover = f"""
    Leg {row['leg_num'] + 1}<br>
    {row['Origin']} → {row['Dest']}<br>
    Dep time: {row['dep_ts_actual']}<br>
    """

    if dep_delay is not None:
        hover += f"Dep delay: {dep_delay:.1f} min<br>"
    if prev_arr_delay is not None:
        hover += f"Prev arr delay: {prev_arr_delay:.1f} min<br>"
    if turnaround is not None:
        hover += f"Turnaround: {turnaround:.1f} min<br>"

    width = 2
    if prev_arr_delay is not None and prev_arr_delay > 15:
        width = 5

    fig.add_trace(
        go.Scattergeo(
            lon=[lon0, lon1],
            lat=[lat0, lat1],
            mode="lines+markers+text",
            text=[row["Origin"], row["Dest"]],
            textposition="top center",
            line=dict(width=width),
            marker=dict(size=7),
            hoverinfo="text",
            textfont=dict(size=10),
            name=f"Leg {row['leg_num'] + 1}",
            hovertext=hover,
        )
    )

fig.update_layout(
    title=f"Aircraft Rotation Propagation Story: Tail {TAIL_TO_VIEW}",
    geo=dict(
        scope="usa",
        projection_type="albers usa",
        showland=True,
        showlakes=True,
        showocean=True,
    ),
    height=800,
    showlegend=False,
)

fig.show()